### ByT5 Implementation for the Akkadian translation

In [40]:
# !pip install -q transformers datasets evaluate sacrebleu sentencepiece accelerate
# !pip install --upgrade torch
# !pip install --upgrade torch torchvision transformers
print("All required libraries are installed and up to date.")


All required libraries are installed and up to date.


## imports

In [41]:
import os
import numpy as np 
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from transformers import ( 
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)
import evaluate

## configurations

In [42]:
MODEL_NAME = "google/byt5-small"
TASK_PREFIX = "translation_akkadian_to_english: "

MAX_INPUT_LEN = 512
MAX_TARGET_LEN = 256

BATCH_SIZE = 2
GRAD_ACCUM = 4      
NUM_EPOCHS = 5
LR = 1e-4
OUTPUT_DIR = "./byt5_akkadian"

TRAIN_CSV = "../data/cleaned/cleaned_train.csv"
VAL_CSV   = "../data/cleaned/cleaned_val.csv"

print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Model : {MODEL_NAME}")

Device: NVIDIA GeForce RTX 2050
Model : google/byt5-small


## load and inspect data

In [43]:
df_train = pd.read_csv(TRAIN_CSV)
df_val = pd.read_csv(VAL_CSV)

print(f"Train: {len(df_train)} rows | Val: {len(df_val)} rows")
print("\nColumns:", df_train.columns.tolist())

# Keep only what we need
df_train = df_train[["transliteration", "translation"]].dropna()
df_val   = df_val[["transliteration", "translation"]].dropna()

print("\n--- Sample ---")
print("INPUT :", df_train.iloc[1]["transliteration"][:120])
print("TARGET:", df_train.iloc[1]["translation"][:120])

Train: 1404 rows | Val: 157 rows

Columns: ['oare_id', 'transliteration', 'translation', 'translit_len', 'transl_len', 'len_ratio', 'flag_very_short_translation', 'flag_very_long_translation', 'flag_low_ratio', 'flag_has_gap']

--- Sample ---
INPUT : 16 ma-na KÙ ṣa-ru-pá-am i-ṣé-er a-šùr-eš-tí-kál ù i-dí-a-bi₄-im i-sú-rik₁₃ i-šu IGI a-ku-za IGI ib-ni-sú-in
TARGET: Aššuriš-tikal and Iddin-abum owe 16 minas of refined silver to Issu-arik. Witnessed by Akuza, witnessed by Ibni-Suen.


In [44]:
raw_datasets = DatasetDict({
    "train": Dataset.from_pandas(df_train, preserve_index=False),
    "validation": Dataset.from_pandas(df_val, preserve_index=False),
})

print(raw_datasets)

DatasetDict({
    train: Dataset({
        features: ['transliteration', 'translation'],
        num_rows: 1404
    })
    validation: Dataset({
        features: ['transliteration', 'translation'],
        num_rows: 157
    })
})


## tokenizer & preprocessing

In [45]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess(examples):
    inputs = [TASK_PREFIX + t for t in examples["transliteration"]]
    targets = examples["translation"]
    
    model_inputs = tokenizer(inputs, max_length = MAX_INPUT_LEN, truncation=True, padding=False)
    
    
    labels = tokenizer(text_target=targets, max_length = MAX_TARGET_LEN, truncation=True, padding=False)
        
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized = raw_datasets.map(preprocess, batched=True, remove_columns=["transliteration", "translation"])

# sanity check 
sample_ids = tokenized["train"][0]["input_ids"]
print("\nDecoded sample input (first 200 bytes):", tokenizer.decode(sample_ids[:200]))

Map:   0%|          | 0/1404 [00:00<?, ? examples/s]

Map:   0%|          | 0/157 [00:00<?, ? examples/s]


Decoded sample input (first 200 bytes): translation_akkadian_to_english: ṭup-pu-um ša ba-áb DINGIR ša kà-nu-tim a-ša-at ili₅-ba-ni 2 na-áš-pé-er-tum ša a-ta-ta DUMU ma-num-ba-lúm-a-šùr a-na ṣé-ri-a ù a-ṣé-er a-šùr-e


## load model

In [46]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

num_params = sum(p.numel() for p in model.parameters())/1e6
model.config.tie_word_embeddings = False  # fixes the missing keys warning
print(f"\nModel has {num_params:.2f} million parameters.")

Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



Model has 299.64 million parameters.


## metrics

In [47]:
bleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    
    if isinstance(preds, tuple):
        preds = preds[0]
        
    preds = np.clip(preds, 0, tokenizer.vocab_size - 1)
        
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    
    
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]
    
    bleu = bleu_metric.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
    chrf = chrf_metric.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels], word_order=2)
    
    return {"bleu": round(bleu["score"], 4), "chrf++": round(chrf["score"], 4)}
    

## training arguments

In [ ]:

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_steps=50,
    weight_decay=0.01,
    lr_scheduler_type="linear",
    
    save_strategy="steps",
    save_steps=100,
    eval_strategy="steps",
    eval_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    greater_is_better=True,
    save_total_limit=3,
    
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=0,
    
    logging_steps=20,
    report_to="none",
    fp16_full_eval=False,    
)
    

SyntaxError: keyword argument repeated: fp16 (3652547552.py, line 29)

## data collator

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100, pad_to_multiple_of=8 if torch.cuda.is_available() else None)

## train

In [ ]:

trainer = Seq2SeqTrainer(
    model=model, 
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)

In [50]:
print(next(model.parameters()).device)

cpu


In [ ]:
trainer.train()

Step,Training Loss,Validation Loss,Bleu,Chrf++
100,1309.252344,nan,0.242700,4.122000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [51]:
import os
print(os.listdir("./byt5_akkadian"))

['checkpoint-100', 'checkpoint-300', 'checkpoint-400']


In [52]:
results = trainer.evaluate()
print("\nValidation results:")
for k, v in results.items():
    print(f"{k}: {v}")

KeyboardInterrupt: 

## generate predicitons & save

In [ ]:
preds_output = trainer.predict(tokenized["validation"])

pred_ids = preds_output.predictions
label_ids = preds_output.label_ids

decoded_preds = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
label_ids = np.where(label_ids != -100, label_ids, tokenizer.pad_token_id)
decoded_labels = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

pred_df = pd.DataFrame({
    "transliteration":  df_val["transliteration"].values,
    "ground_truth": [l.strip() for l in decoded_labels],
    "prediction": [p.strip() for p in decoded_preds],
})

pred_df.to_csv("byt5_predictions.csv", index=False)
print(f"\n {len(pred_df)} predictions saved to byt5_predictions.csv")

# examples:
for i in range(3):
    print(f"\nExample {i+1}:")
    print("SRC:", pred_df.iloc[i]["transliteration"][:100])
    print("REC:", pred_df.iloc[i]["ground_truth"][:150])
    print("HYP:", pred_df.iloc[i]["prediction"][:150])


## save model checkpoint

In [ ]:
trainer.save_model(OUTPUT_DIR + "/best_model")
tokenizer.save_pretrained(OUTPUT_DIR + "/best_model")
print(f"\nModel and tokenizer saved to {OUTPUT_DIR}/best_model")

## quick inference test

In [ ]:
def translate(akkadian_text: str, max_new_tokens: int = 256) -> str:
    model.eval()
    
    input_text = TASK_PREFIX + akkadian_text
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LEN).to(model.device)
    
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_length=max_new_tokens,
            num_beams=4,
            early_stopping=True
        )
    
    return tokenizer.decode(generated_ids[0], skip_special_tokens=True)

sample = df_val.iloc[0]["transliteration"]
print("\nSample Akkadian:", sample[:200])
print("\nPREDICTION:", translate(sample))
print("\nGROUND TRUTH:", df_val.iloc[0]["translation"][:200])

In [ ]:
import os
for f in os.listdir("./byt5_akkadian"):
    print(f)